In [ ]:
# Only run this once at the beginning of the notebook to set up the correct working directory and import paths
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent  # because notebook is inside data_preprocessing
sys.path.append(str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print(f"Current working directory: {os.getcwd()}")

## Load Model

In [ ]:
import torch
from utils.model_utils import get_model

MODEL_PATHS = [
    "logdir_32_32_gdmc25", # 1000 steps
    "logdir_32_32_3DCraft_full_grouped2", # 1500 steps
    "logdir_64_64_3DCraft_full_grouped3", # 2000 steps
    "logdir_64_64_3DCraft_full_grouped4", # 2500 steps
]

model_name = "fqgan"
model_path = f"logdir_64_64_3DCraft_full_grouped4"
model_checkpoint_step = 2500

device = 'cuda' if torch.cuda.is_available() else 'cpu'
fqgan, fqgan_hparams = get_model(model_name, model_path, model_checkpoint_step, device=device)

## Plot stats

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from utils.log_utils import load_stats


# Load training stats for the specified model and checkpoint
fqgan_hparams.load_dir = model_path
steps = 1250
train_stats = load_stats(fqgan_hparams, steps)
steps = 2500
train_stats_2 = load_stats(fqgan_hparams, steps)
# print(train_stats)

save_dir = f"notebooks/loss_plots/{model_path}/{model_name}_{steps}"
os.makedirs(save_dir, exist_ok=True)

# Extract relevant data from stats
# train_stats = {
#                 'losses': loss_arrays,
#                 'codebook_usage': codebook_usage,
#                 'steps_per_log': H.steps_per_log,
#                 'val_losses': val_loss_arrays,
#                 'best_val_loss': best_overall_val_loss,
#                 'steps_per_validation': H.steps_per_validation,
#             }
# plotted_loss_terms = [
#     'rec_loss', 'binary_rec_loss', 'style_loss', 'struct_loss', 
#     'biome_feat_loss', 'disent_loss', 'g_loss', 'd_loss', 'struct_consistency_loss', 'cycle_consistency_loss'
# ]

# losses = train_stats["losses"]
# val_losses = train_stats.get("val_losses", {})
# codebook_usage = train_stats.get("codebook_usage", {})
# best_val_loss = train_stats.get("best_val_loss", None)
# steps_per_log = train_stats["steps_per_log"]
# steps_per_validation = train_stats["steps_per_validation"]
def merge_series_dicts(first, second):
    merged = {}

    for key in first.keys() | second.keys():
        values_1 = first.get(key, [])
        values_2 = second.get(key, [])

        merged[key] = np.concatenate([
            np.asarray(values_1),
            np.asarray(values_2),
        ])

    return merged
losses = merge_series_dicts(train_stats["losses"], train_stats_2["losses"])
val_losses = merge_series_dicts(train_stats.get("val_losses", {}), train_stats_2.get("val_losses", {}))
codebook_usage = merge_series_dicts(train_stats.get("codebook_usage", {}), train_stats_2.get("codebook_usage", {}))
best_val_loss = train_stats.get("best_val_loss", None)
steps_per_log = train_stats["steps_per_log"]
steps_per_validation = train_stats["steps_per_validation"]


def plot_loss_pair(term, title, save_path=None):
    plt.figure(figsize=(9, 5))

    if term in losses and len(losses[term]) > 0:
        y = losses[term]
        x = np.arange(1, len(losses[term]) + 1) * steps_per_log
        plt.plot(x, y, label=f"train")

    if term in val_losses and len(val_losses[term]) > 0:
        y = val_losses[term]
        x = np.arange(1, len(val_losses[term]) + 1) * steps_per_validation
        plt.plot(x, y, label=f"validation")

    plt.xlabel("Training step")
    plt.ylabel("Loss")
    plt.title(title, size=18)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path)
    plt.show()
    

# 1. Reconstruction
plot_loss_pair("rec_loss", "Reconstruction Loss", save_path=f"{save_dir}/reconstruction_loss.pdf")

# 2. Auxiliary losses
plotted_loss_terms = ["binary_rec_loss", "disent_loss", "g_loss"]
plotted_loss_titles = {
    "binary_rec_loss": "Binary Reconstruction Loss",
    "disent_loss": "Disentanglement Loss",
    "g_loss": "Generator Loss"
}
for term in plotted_loss_terms:
    plot_loss_pair(term, f"{plotted_loss_titles.get(term, 'UNKNOWN_TITLE')}", save_path=f"{save_dir}/{term}.pdf")


# 3. Codebook usage
plt.figure(figsize=(9, 5))
for name in ["style", "struct"]:
    if name in codebook_usage and len(codebook_usage[name]) > 0:
        y = [value*100 for value in codebook_usage[name]]
        x = np.arange(1, len(codebook_usage[name]) + 1) * steps_per_log
        plt.plot(x, y, label=f"{name} codebook usage", color='C2' if name == "style" else 'C3')

plt.xlabel("Training step")
plt.ylabel("Usage (%)")
plt.title("Codebook usage", size=18)
plt.legend()
plt.grid(True)
plt.tight_layout()

plt.savefig(f"{save_dir}/codebook_usage.pdf")

plt.show()


# 3. Codebook losses
# plt.figure(figsize=(9, 5))
# for term in ["style_loss", "struct_loss"]:
#     if term in losses and len(losses[term]) > 0:
#         y = losses[term]
#         x = np.arange(1, len(losses[term]) + 1) * steps_per_log
#         plt.plot(x, y, label=f"train {term}")

#     if term in val_losses and len(val_losses[term]) > 0:
#         y = val_losses[term]
#         x = np.arange(1, len(val_losses[term]) + 1) * steps_per_validation
#         plt.plot(x, y, label=f"val {term}")

# plt.xlabel("training step")
# plt.ylabel("loss")
# plt.title("Style and structure codebook losses")
# plt.legend()
# plt.grid(True)
# plt.tight_layout()

# plt.savefig(f"{save_dir}/codebook_losses.png")

# plt.show()


